## 1. The Generator: the atomic unit of streaming

### a. Normal (synchronous) generator
- `yield` is what makes this a generator. The function body doesn't execute when called — it returns a generator object. Execution runs only when the caller pulls via `next()` or iteration. This is lazy evaluation: work happens on demand, not upfront.

In [14]:
import time


def token_stream():
    for token in ["Hello", " world", "!"]:
        time.sleep(0.2)
        yield token  #  suspends the generator, sends token to the caller, returns control to the caller

# Runs on next():
gen = token_stream() # nothing runs
print(next(gen), end="") # runs until the 1st yield is reached
print(next(gen), end="") # resumes from suspension point
print(next(gen), end="")
# print(next(gen)) # 4th time runs out of token, returns error `StopIteration`

# Runs on iteration:
print()
gen = token_stream()
for token in gen:
    print(token, end="", flush=True)

Hello world!
Hello world!

### b. Asynchronous Generator
- `yield` is still present. Each generation runs at each `await anext()`, or when iterated with `async for` loop.

In [ ]:
import asyncio

async def atoken_stream():
    for token in ["Hello", " world", "!"]:
        await asyncio.sleep(0.2)
        yield token

# Run on anext():
agen = atoken_stream()
print(await anext(agen), end="")
print(await anext(agen), end="")
print(await anext(agen), end="")

# Run on iteration:
print()
agen = atoken_stream()
async for token in agen:
    print(token, end="")

Hello world!
Hello world!

Read more in the `streaming-advanced` notebook

## 2. Flushing
- To display the stream in the intended behavior, where each token appears after the previous one, use `flush=True`.
    - When you call print("hello"), Python doesn't necessarily write "hello" directly to the terminal. Instead, it writes to an in-memory buffer.
    - When your Python process is running inside a web server, Docker container, or Streamlit's subprocess — stdout is piped, not a TTY. So Python switches to fully buffered mode, meaning tokens accumulate silently until the buffer fills up.
    - `flush()` tells libc: don't wait, push the buffer to the OS right now. The OS then writes to whatever the file descriptor points to — terminal, pipe, socket.
- This behavior is seen when running the script `flush.py` in the terminal, not IPYNB. Jupyter is actually line-buffered by default, flushing on \n. If the streaming loop has no \n per token, flush=True vs not shouldn't matter — both get buffered by the 100ms timer and flushed together.


## 3. Generator chaining: streaming pipelines

- Generators compose naturally into pipelines where each innter stage processes tokens as they arrive, and passes to the outer stage:

In [ ]:
import time
from collections.abc import Generator


def source():
    """Simumates an LLM that emits tokens with latency"""
    for token in ["The", " answer", " is", " 42", "."]:
        time.sleep(0.3)
        yield token

def uppercase_filter(stream: Generator) -> Generator:
    """Transforms tokens as they pass through."""
    for token in stream:
        yield token.upper()

def prefix_filter(stream: Generator, prefix:str=">> ") -> Generator:
    """Adds metadata - simulates tagging node outputs."""
    first = True
    for token in stream:
        if first:
            yield prefix
            first = False
        yield token

# Pipeline:
pipeline = prefix_filter(uppercase_filter(source()))

for token in pipeline:
    print(token, end="", flush=True)

>> THE ANSWER IS 42.

## 4. Generator Advance

### Channel 1: values out from generator to caller
- This is the most common use of generator.
- the value after "yield" is sent to the caller every iteration.
- there is no need to prime the generator

In [22]:
def thinking_stream():
    yield "Thinking"
    time.sleep(0.5)
    yield "..."
    time.sleep(2)

def answer_stream():
    for token in ["The", " answer", " is", " 42", "."]:
        time.sleep(0.3)
        yield token

# Boiler-plate: runs by using the iterator (for loop) in the main generator
def agent_stream():
    for token in thinking_stream():   # manual delegation
        yield token
    for token in answer_stream():     # manual delegation
        yield token

print("Manual delegation:")
for token in agent_stream():
    print(token, end="", flush=True)
print("\n")

# Using `yield from` in the main generator:
print("Using yield from:")
def agent_stream():
    yield from thinking_stream()
    yield from answer_stream()

for token in agent_stream():
    print(token, end="", flush=True)

Manual delegation:
Thinking...The answer is 42.

Using yield from:
Thinking...The answer is 42.

### Channel 2: values into the generator with `send`
- Full expression: 
    ```python
    def gen():
        received = yield emitted
    ```
- Every yield expression has two jobs:
    - right hand side: suspends, then emits that value to the caller
    - left hand side: once caller resumes and sends a value, assign it to this
- The caller sends the value via the `generator.send(<value>)` method.
    - If the sent value is None, it is treated as `generator.next()`
    - The behavior of the generator vs the values sent can be freely controlled.
- Once the generator stops, e.g., when finishing the tokens or hitting a condition, it will raise the `StopIteration` exception.
    - It's not an error — it's the standard protocol for "generator is done."
    - This should be handled by the cliend side.
- Misconception: Priming is needed to advance the generator past the first `received` and suspends at the first `yield` (left to right)
- Correction: Priming advances the generator to the first `yield`, sends `emitted` to caller, suspends there. The `received` has not been assigned yet, that happends only when `send()` resumes it.
    - though `emitted` is sent to caller during priming, it may be silent if it is not printed out or assigned to anything

In [ ]:
def controlled_stream():
    """
    Emits tokens until:
    - user sends "stop": exit
    - user sends "slow": adds 1s in between tokens.
    """
    tokens = ["The", " answer", " to", " life", " is", " 42", "."]
    for token in tokens:
        # when the generator resumes, the value that resumed it (via send()) is assigned to `command`
        # if there is none sent from the caller, yield will continue to send the token to the caller
        command = yield token

        if command == "stop":
            print("\n[stream cancelled]")
            return

        elif command == "slow":
            print("\n[slowing down]")
            time.sleep(1) # adds delay before continuing to next token

gen = controlled_stream()

# Prime the generator
token = next(gen)
print(token, end="", flush=True)

# The next() above is equivalent to sending None to the generator:
gen = controlled_stream() # restarts
token = gen.send(None) 
print(token, end="", flush=True)

# Normal consumption for the next 3 tokens:
for _ in range(3):
    token = gen.send(None)
    print(token, end="", flush=True)

# Signal slow down and genarate the next token
token = gen.send("slow")
print(token, end="", flush=True)

# Signal stop mid-stream
try:
    gen.send("stop") # raise Exception
except StopIteration:
    pass

TheThe answer to life
[slowing down]
 is
[stream cancelled]


In [ ]:
# a function that receives tokens via send(), yields running count stats after each one.
def token_accumulator():
    count = 0
    while True:
        # suspends, send count to caller first, then assign sender's value to command
        command = yield count 
        
        # if command is empty, exit
        if command == None:
            # this will raise StopIteration.  
            break 
        
        # if command is not empty, continue
        else:
            count = count + 1

def another_token_accumulator():
    count = 0
    while True:
        command = yield count
        if command == None:
            count += 1
        elif command == "reset":
            count = 0
        elif command == "stop":
            break

# def token_accumulator():
#     token_count = 0
#     while True:
#         token = yield token_count
#         if token is None:
#             break
#         token_count += 1


acc = token_accumulator() # init but frozen
next(acc) # priming, advances to 1st yield and suspend there

stats = acc.send("hi1") # caller sends value, assifned to "command"
print(stats) # now count for the 1st time

stats = acc.send("hi2")
print(stats)

stats = acc.send("hi3")
print(stats)

try:
    stats = acc.send(None)
except StopIteration:
    print("Generator has been stopped.")

acc2 = another_token_accumulator()
next(acc2)

print(acc2.send(None))
print(acc2.send(None))
print(acc2.send(None))

print(acc2.send("reset")) # returns the RHS of the yield statement = 0
print(acc2.send(None)) # continues to 1

try:
    acc2.send("stop")
except StopIteration:
    print("Generator has been stopped.")

1
2
3
Generator has been stopped.
1
2
3
0
1
Generator has been stopped.


### Channel 3: Exceptions in - from caller to generator

In [ ]:
# TBU

## 5. Using `yield from`
- when a sub-generator is called inside another function (a generator as well), manually iterating the sub- with for-yield loop only uses the output channel. `send()` and `throw()` never reach the sub-
- `yield from` wires all three channels transparently

In [85]:
# Expected behavior:
# sub() emits token1, receives command, prints out, then emits token2

def sub():
    command = yield "token1"
    print(f"sub received: {command}")
    yield "token2"

def manual_delegator():
    for token in sub():
        yield token

gen = manual_delegator()
next(gen) # prime as the child needs priming, this produces "token1"
gen.send("hi")

sub received: None


'token2'

In [86]:
def proper_delegator():
    yield from sub()

gen = proper_delegator()
next(gen)
gen.send("hi")

sub received: hi


'token2'